# LeetCode 288 — Unique Word Abbreviation bằng Python

**Link đề:** https://leetcode.com/problems/unique-word-abbreviation/
**Độ khó:** Easy (theo danh mục repo) · **Chủ đề:** Array, Hash Table, String, Design

> Ghi chú độ khó: LeetCode hiện ghi bài này là **Medium**; repo xếp vào **Easy** nên notebook lưu trong `DSA/Easy`.

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class ValidWordAbbr:

    def __init__(self, dictionary: List[str]):
        ...

    def isUnique(self, word: str) -> bool:
        ...

# Your ValidWordAbbr object will be instantiated and called as such:
# obj = ValidWordAbbr(dictionary)
# param_1 = obj.isUnique(word)
```

## Mục tiêu học
1. Đọc kỹ một đề **thiết kế lớp** (design): hàm khởi tạo xử lý trước, hàm truy vấn gọi nhiều lần.
2. Viết hàm **viết tắt** (abbreviation) đúng mọi độ dài: `"internationalization" → "i18n"`, `"it" → "it"`.
3. Phân tích chính xác điều kiện "unique" — chỗ **dễ hiểu sai nhất** của bài (từ trùng trong từ điển, từ hỏi nằm sẵn trong từ điển).
4. Đi từ quét toàn bộ từ điển mỗi truy vấn → **tiền xử lý bằng `dict`** để mỗi truy vấn `O(L)`.
5. So sánh 4 cách lưu trữ: quét thẳng, `dict → set`, `dict → từ đại diện` (sentinel), `Counter` + `set`.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

**Viết tắt** của một từ = `chữ cái đầu` + `số ký tự ở giữa` + `chữ cái cuối`. Nếu từ chỉ có **2 ký tự trở xuống** thì viết tắt chính là từ đó.

| Từ | Viết tắt |
|---|---|
| `"dog"` | `"d1g"` (giữa có 1 ký tự `o`) |
| `"internationalization"` | `"i18n"` |
| `"it"` | `"it"` |
| `"a"` | `"a"` |

Cài đặt lớp `ValidWordAbbr`:
- `ValidWordAbbr(dictionary)`: khởi tạo với danh sách từ.
- `isUnique(word)`: trả `True` nếu **một trong hai** điều kiện đúng:
  1. **Không** có từ nào trong `dictionary` có cùng viết tắt với `word`;
  2. **Mọi** từ trong `dictionary` có cùng viết tắt với `word` đều **chính là** `word`.

**Ví dụ:** `dictionary = ["deer", "door", "cake", "card"]`

| Truy vấn | Kết quả | Vì sao |
|---|:---:|---|
| `isUnique("dear")` | `False` | `"d2r"` trùng với `"deer"`, `"door"` (khác `"dear"`) |
| `isUnique("cart")` | `True` | `"c2t"` không có trong từ điển |
| `isUnique("cane")` | `False` | `"c2e"` trùng với `"cake"` |
| `isUnique("make")` | `True` | `"m2e"` không có |
| `isUnique("cake")` | `True` | chỉ `"cake"` có `"c2e"` — chính là nó |

**Constraints (theo hiểu biết về đề):** `1 <= len(dictionary) <= 3·10^4`; mỗi từ dài `1..20`, chữ thường; `word` dài `1..20`; tối đa `5000` lần gọi `isUnique`.

## 2. Nền tảng Python

### 2.1. Truy cập ký tự, `len`, `str()` và ghép chuỗi

- `w[0]` ký tự đầu, `w[-1]` ký tự cuối (chỉ số âm đếm từ cuối).
- `len(w) - 2` = số ký tự ở giữa.
- `str(18)` → `"18"`; `+` nối chuỗi. Hoặc dùng **f-string**: `f"{w[0]}{len(w) - 2}{w[-1]}"`.

In [1]:
w = "internationalization"
print(w[0], w[-1], len(w), len(w) - 2)
print(w[0] + str(len(w) - 2) + w[-1])
print(f"{w[0]}{len(w) - 2}{w[-1]}")

i n 20 18
i18n
i18n


In [2]:
def abbr(w: str) -> str:
    if len(w) <= 2:
        return w
    return f"{w[0]}{len(w) - 2}{w[-1]}"

for w in ["dog", "internationalization", "it", "a", "deer", "door", "dear", "cake", "card"]:
    print(f"{w!r:>24} -> {abbr(w)!r}")

                   'dog' -> 'd1g'
  'internationalization' -> 'i18n'
                    'it' -> 'it'
                     'a' -> 'a'
                  'deer' -> 'd2r'
                  'door' -> 'd2r'
                  'dear' -> 'd2r'
                  'cake' -> 'c2e'
                  'card' -> 'c2d'


**Vì sao phải xử lý riêng `len <= 2`?** Nếu áp công thức cho `"it"` ta được `"i0t"`, cho `"a"` ta được `"a-1a"` — vô nghĩa. Đề quy định viết tắt của từ ngắn là chính nó.

In [3]:
def abbr_naive(w):
    return f"{w[0]}{len(w) - 2}{w[-1]}"
print(abbr_naive("it"), abbr_naive("a"))

i0t a-1a


### 2.2. `dict` với `setdefault`, `set`, `defaultdict`

- `d.setdefault(k, set())` trả về `d[k]`, nếu chưa có thì tạo `set()` rồi trả về → tiện để `.add(...)`.
- `collections.defaultdict(set)`: dict tự tạo `set()` khi truy cập key mới.
- `set` tự loại phần tử trùng.

In [4]:
from collections import defaultdict, Counter
groups = {}
for w in ["deer", "door", "cake", "card", "deer"]:
    groups.setdefault(abbr(w), set()).add(w)
print(groups)

g2 = defaultdict(set)
for w in ["deer", "door", "cake"]:
    g2[abbr(w)].add(w)
print(dict(g2), "| key chưa có:", g2["zzz"], "-> dict giờ có key 'zzz':", "zzz" in g2)

{'d2r': {'door', 'deer'}, 'c2e': {'cake'}, 'c2d': {'card'}}
{'d2r': {'door', 'deer'}, 'c2e': {'cake'}} | key chưa có: set() -> dict giờ có key 'zzz': True


Lưu ý cuối: **chỉ đọc** `g2["zzz"]` cũng tạo key mới trong `defaultdict`. Khi truy vấn, nên dùng `g2.get(key)` để không làm "bẩn" cấu trúc.

### 2.3. Lớp (class) trong Python — đủ dùng cho bài design

- `__init__(self, ...)` chạy khi tạo đối tượng; gán `self.x = ...` để lưu dữ liệu cho các lần gọi sau.
- Các method khác dùng lại `self.x`.

In [5]:
class Counter2:
    def __init__(self, start):
        self.value = start
    def inc(self):
        self.value += 1
        return self.value

c = Counter2(10)
print(c.inc(), c.inc(), c.value)

11 12 12


## 3. Phân tích điều kiện "unique" — đọc chậm từng chữ

Đặt `A = abbr(word)` và `S = {các từ trong dictionary có viết tắt A}` (tập, đã bỏ trùng).

- Điều kiện 1: `S` rỗng.
- Điều kiện 2: mọi từ trong `S` đều bằng `word` ⇔ `S ⊆ {word}`.

Gộp lại: **`isUnique(word)` ⇔ `S ⊆ {word}`** (tập rỗng cũng là tập con).

Các tình huống dễ nhầm:

| dictionary | word | S | Kết quả |
|---|---|---|:---:|
| `["cake"]` | `"cake"` | `{cake}` | `True` (từ hỏi nằm sẵn trong từ điển) |
| `["cake", "cake"]` | `"cake"` | `{cake}` | `True` (từ điển có **trùng lặp**) |
| `["cake", "cane"]` | `"cake"` | `{cake, cane}` | `False` |
| `["a"]` | `"a"` | `{a}` | `True` |
| `["a"]` | `"b"` | `{}` | `True` (viết tắt `"b"` khác `"a"`) |

In [6]:
def is_unique_by_definition(dictionary, word):
    S = {w for w in dictionary if abbr(w) == abbr(word)}     # set comprehension
    return S <= {word}

for d, w in [(["cake"], "cake"), (["cake", "cake"], "cake"), (["cake", "cane"], "cake"), (["a"], "a"), (["a"], "b")]:
    print(f"dictionary = {d}, word = {w!r} -> {is_unique_by_definition(d, w)}")

dictionary = ['cake'], word = 'cake' -> True
dictionary = ['cake', 'cake'], word = 'cake' -> True
dictionary = ['cake', 'cane'], word = 'cake' -> False
dictionary = ['a'], word = 'a' -> True
dictionary = ['a'], word = 'b' -> True


## 4. Cách 1 — Baseline: quét toàn bộ từ điển mỗi lần hỏi

`__init__` chỉ lưu danh sách. `isUnique` duyệt từng từ: nếu cùng viết tắt mà **khác** `word` → `False` ngay.

Chi phí mỗi truy vấn `O(N · L)` (`N` từ, mỗi từ tính viết tắt `O(L)` vì tạo chuỗi số). Với `N = 3·10^4` và `5000` truy vấn → `1.5·10^8` lần tính viết tắt: quá chậm cho Python.

In [7]:
class ValidWordAbbrBrute:
    def __init__(self, dictionary: list[str]):
        self.words = dictionary

    def isUnique(self, word: str) -> bool:
        a = abbr(word)
        for w in self.words:
            if abbr(w) == a and w != word:
                return False
        return True

obj = ValidWordAbbrBrute(["deer", "door", "cake", "card"])
print([obj.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]])

[False, True, False, True, True]


### 4.1. Nút thắt và insight — **tiền xử lý** một lần, trả lời nhiều lần

Từ điển **không đổi** giữa các truy vấn. Vậy hãy làm việc nặng **một lần** trong `__init__`: nhóm các từ theo viết tắt.

```text
"d2r" -> {"deer", "door"}
"c2e" -> {"cake"}
"c2d" -> {"card"}
```

Khi hỏi `word`, chỉ cần tính `abbr(word)` rồi tra **một** key trong dict — `O(L)` thay vì `O(N·L)`. Đây là mẫu thiết kế "**precompute + hash lookup**".

## 5. Cách 2 — `dict`: viết tắt → `set` các từ

**Đoạn 1 — `__init__`:**
```python
self.groups = {}
for w in dictionary:
    self.groups.setdefault(abbr(w), set()).add(w)
```

**Đoạn 2 — `isUnique`:**
```python
S = self.groups.get(abbr(word), set())   # không có -> tập rỗng
return S <= {word}                        # rỗng, hoặc chỉ chứa chính word
```

Rõ ràng, bám sát định nghĩa ở mục 3. Nhược điểm: lưu **mọi** từ (bộ nhớ `O(N·L)`).

In [8]:
class ValidWordAbbrSets:
    def __init__(self, dictionary: list[str]):
        self.groups = {}
        for w in dictionary:
            self.groups.setdefault(abbr(w), set()).add(w)

    def isUnique(self, word: str) -> bool:
        S = self.groups.get(abbr(word), set())
        return S <= {word}

obj = ValidWordAbbrSets(["deer", "door", "cake", "card"])
print(obj.groups)
print([obj.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]])

{'d2r': {'door', 'deer'}, 'c2e': {'cake'}, 'c2d': {'card'}}
[False, True, False, True, True]


## 6. Cách 3 — Lời giải chính: viết tắt → **một từ đại diện** (hoặc dấu "nhiều")

Quan sát: `isUnique` chỉ cần biết `S` là **rỗng**, **đúng một từ** (và từ đó là gì), hay **từ hai từ khác nhau trở lên**. Không cần lưu cả tập.

Lưu `self.owner[a]`:
- chưa có key `a` → `S` rỗng;
- `owner[a] = w` → `S = {w}`;
- `owner[a] = ""` (chuỗi rỗng — **sentinel**, không thể là một từ hợp lệ vì từ dài ≥ 1) → có ≥ 2 từ khác nhau.

**Đoạn 1 — `__init__`:**
```python
for w in dictionary:
    a = abbr(w)
    if a not in self.owner:
        self.owner[a] = w          # từ đầu tiên có viết tắt a
    elif self.owner[a] != w:
        self.owner[a] = ""         # gặp từ thứ hai KHÁC -> đánh dấu "nhiều"
    # owner[a] == w: từ trùng lặp, bỏ qua
```

**Đoạn 2 — `isUnique`:**
```python
a = abbr(word)
return a not in self.owner or self.owner[a] == word
```
Nếu `owner[a] == ""` thì không bằng `word` (vì `word` khác rỗng) → `False`, đúng ý.

In [9]:
class ValidWordAbbrOwner:
    def __init__(self, dictionary: list[str]):
        self.owner = {}
        for w in dictionary:
            a = abbr(w)
            if a not in self.owner:
                self.owner[a] = w
            elif self.owner[a] != w:
                self.owner[a] = ""

    def isUnique(self, word: str) -> bool:
        a = abbr(word)
        return a not in self.owner or self.owner[a] == word

obj = ValidWordAbbrOwner(["deer", "door", "cake", "card"])
print(obj.owner)
print([obj.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]])

{'d2r': '', 'c2e': 'cake', 'c2d': 'card'}
[False, True, False, True, True]


## 7. TRACE từng bước

### 7.1. Trace `__init__` với `["deer", "door", "cake", "card", "cake", "deer"]`

| w | abbr | owner[abbr] trước | hành động | owner[abbr] sau |
|---|---|---|---|---|
| deer | d2r | (chưa có) | gán | deer |
| door | d2r | deer | khác → "nhiều" | "" |
| cake | c2e | (chưa có) | gán | cake |
| card | c2d | (chưa có) | gán | card |
| cake | c2e | cake | trùng, bỏ qua | cake |
| deer | d2r | "" | khác "" → vẫn "" | "" |

In [10]:
def trace_init(dictionary):
    owner = {}
    for w in dictionary:
        a = abbr(w)
        before = owner.get(a, "(chưa có)")
        if a not in owner:
            owner[a] = w
            act = "gán"
        elif owner[a] != w:
            owner[a] = ""
            act = "khác -> 'nhiều'"
        else:
            act = "trùng, bỏ qua"
        print(f"  w = {w:<5} abbr = {a:<4} | trước = {before!r:<12} | {act:<16} | sau = {owner[a]!r}")
    return owner

owner = trace_init(["deer", "door", "cake", "card", "cake", "deer"])
print("owner cuối:", owner)

  w = deer  abbr = d2r  | trước = '(chưa có)'  | gán              | sau = 'deer'
  w = door  abbr = d2r  | trước = 'deer'       | khác -> 'nhiều'  | sau = ''
  w = cake  abbr = c2e  | trước = '(chưa có)'  | gán              | sau = 'cake'
  w = card  abbr = c2d  | trước = '(chưa có)'  | gán              | sau = 'card'
  w = cake  abbr = c2e  | trước = 'cake'       | trùng, bỏ qua    | sau = 'cake'
  w = deer  abbr = d2r  | trước = ''           | khác -> 'nhiều'  | sau = ''
owner cuối: {'d2r': '', 'c2e': 'cake', 'c2d': 'card'}


### 7.2. Trace các truy vấn

In [11]:
def trace_query(owner, word):
    a = abbr(word)
    if a not in owner:
        verdict, why = True, "không ai có viết tắt này"
    elif owner[a] == word:
        verdict, why = True, f"chỉ có chính {word!r}"
    elif owner[a] == "":
        verdict, why = False, "có >= 2 từ khác nhau"
    else:
        verdict, why = False, f"trùng với {owner[a]!r}"
    print(f"  isUnique({word!r:<7}) abbr = {a:<4} -> {verdict!s:<5} ({why})")
    return verdict

for w in ["dear", "cart", "cane", "make", "cake", "deer"]:
    trace_query(owner, w)

  isUnique('dear' ) abbr = d2r  -> False (có >= 2 từ khác nhau)
  isUnique('cart' ) abbr = c2t  -> True  (không ai có viết tắt này)
  isUnique('cane' ) abbr = c2e  -> False (trùng với 'cake')
  isUnique('make' ) abbr = m2e  -> True  (không ai có viết tắt này)
  isUnique('cake' ) abbr = c2e  -> True  (chỉ có chính 'cake')
  isUnique('deer' ) abbr = d2r  -> False (có >= 2 từ khác nhau)


## 8. Cách 4 — `Counter` số từ **phân biệt** theo viết tắt + `set` từ điển

Một cách tư duy khác: đếm mỗi viết tắt có **bao nhiêu từ phân biệt**, và giữ `set` các từ trong từ điển.

`isUnique(word)` ⇔ `count[a] == 0` **hoặc** (`count[a] == 1` **và** `word` có trong từ điển).

Chú ý phải đếm trên `set(dictionary)` — đếm trên list có trùng lặp sẽ sai (xem mục lỗi).

In [12]:
class ValidWordAbbrCounter:
    def __init__(self, dictionary: list[str]):
        self.words = set(dictionary)
        self.count = Counter(abbr(w) for w in self.words)

    def isUnique(self, word: str) -> bool:
        c = self.count.get(abbr(word), 0)
        return c == 0 or (c == 1 and word in self.words)

obj = ValidWordAbbrCounter(["deer", "door", "cake", "card"])
print(obj.count)
print([obj.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]])

Counter({'d2r': 2, 'c2e': 1, 'c2d': 1})
[False, True, False, True, True]


Vì sao `c == 1 and word in self.words` là đủ? Nếu chỉ có **một** từ phân biệt mang viết tắt `a`, và `word` (cũng mang viết tắt `a`) nằm trong từ điển, thì từ duy nhất đó **phải** là `word`.

## 9. Code nộp LeetCode

Dán class sau (LeetCode đã `from typing import List`). Hàm viết tắt đặt thành method tĩnh để class tự đủ.

In [13]:
from typing import List

class ValidWordAbbr:

    def __init__(self, dictionary: List[str]):
        self.owner = {}
        for w in dictionary:
            a = self._abbr(w)
            if a not in self.owner:
                self.owner[a] = w
            elif self.owner[a] != w:
                self.owner[a] = ""

    @staticmethod
    def _abbr(w: str) -> str:
        return w if len(w) <= 2 else f"{w[0]}{len(w) - 2}{w[-1]}"

    def isUnique(self, word: str) -> bool:
        a = self._abbr(word)
        return a not in self.owner or self.owner[a] == word


obj = ValidWordAbbr(["deer", "door", "cake", "card"])
print([obj.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]])

[False, True, False, True, True]


`@staticmethod`: method không cần `self` — gọi được qua `self._abbr(...)` hay `ValidWordAbbr._abbr(...)`.

### 9.1. Mô phỏng cách LeetCode chấm bài design

LeetCode đưa hai list: tên thao tác và tham số; output là list kết quả (`None` cho constructor).

In [14]:
def run_ops(cls, ops, args):
    out, obj = [], None
    for op, arg in zip(ops, args):          # zip ghép từng cặp (op, arg)
        if op == "ValidWordAbbr":
            obj = cls(*arg)                 # *arg "mở" list tham số
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))   # getattr lấy method theo tên chuỗi
    return out

ops = ["ValidWordAbbr", "isUnique", "isUnique", "isUnique", "isUnique", "isUnique"]
args = [[["deer", "door", "cake", "card"]], ["dear"], ["cart"], ["cane"], ["make"], ["cake"]]
print(run_ops(ValidWordAbbr, ops, args))

[None, False, True, False, True, True]


## 10. Vì sao đúng? + độ phức tạp

**Invariant của `__init__` (cách 3):** sau khi xử lý một tiền tố của `dictionary`, với mọi viết tắt `a`:
- `a` không là key ⇔ chưa có từ nào mang viết tắt `a`;
- `owner[a] = w` (khác rỗng) ⇔ tập các từ **phân biệt** mang viết tắt `a` đúng bằng `{w}`;
- `owner[a] = ""` ⇔ tập đó có ≥ 2 phần tử.

Mỗi từ mới chỉ có thể: tạo key (0 → 1), giữ nguyên (trùng), hoặc chuyển 1 → "nhiều"; "nhiều" giữ mãi. Từ đó `isUnique` trả `True` đúng khi `S ⊆ {word}` như mục 3.

Gọi `N` = số từ, `L` = độ dài tối đa (≤ 20), `Q` = số truy vấn.

| Cách | `__init__` | `isUnique` | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Quét thẳng | `O(1)` | `O(N·L)` | `O(1)` thêm | chậm khi nhiều truy vấn |
| 2. `dict → set` | `O(N·L)` | `O(L)` trung bình | `O(N·L)` | rõ ràng nhất |
| 3. `dict → owner` | `O(N·L)` | `O(L)` trung bình | `O(K·L)`, `K` = số viết tắt khác nhau | **lời giải chính** |
| 4. `Counter` + `set` | `O(N·L)` | `O(L)` trung bình | `O(N·L)` | tư duy đếm |

Tổng thời gian cách 3: `O((N + Q)·L)`.

In [15]:
import random, string, time
rng = random.Random(288)
big_dict = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 8))) for _ in range(30000)]
queries = ["".join(rng.choice("abcd") for _ in range(rng.randint(1, 8))) for _ in range(100)]
for cls in [ValidWordAbbrBrute, ValidWordAbbrOwner]:
    t0 = time.perf_counter()
    o = cls(big_dict)
    res = [o.isUnique(q) for q in queries]
    print(f"{cls.__name__:<20}: {time.perf_counter() - t0:.3f}s cho 100 truy vấn | số True = {sum(res)}")

ValidWordAbbrBrute  : 0.294s cho 100 truy vấn | số True = 52
ValidWordAbbrOwner  : 0.007s cho 100 truy vấn | số True = 52


## 11. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<44} PASS ({actual})")

o = ValidWordAbbr(["deer", "door", "cake", "card"])
check("mẫu: dear", o.isUnique("dear"), False)
check("mẫu: cart", o.isUnique("cart"), True)
check("mẫu: cane", o.isUnique("cane"), False)
check("mẫu: make", o.isUnique("make"), True)
check("mẫu: cake", o.isUnique("cake"), True)
check("từ hỏi có trong từ điển nhưng có đồng viết tắt", o.isUnique("deer"), False)
check("từ điển trùng lặp", ValidWordAbbr(["cake", "cake"]).isUnique("cake"), True)
check("trùng lặp + từ khác", ValidWordAbbr(["cake", "cake", "cane"]).isUnique("cake"), False)
check("từ 1 ký tự giống", ValidWordAbbr(["a"]).isUnique("a"), True)
check("từ 1 ký tự khác", ValidWordAbbr(["a"]).isUnique("b"), True)
check("từ 2 ký tự", ValidWordAbbr(["it", "at"]).isUnique("it"), True)
check("'hello' vs 'hallo' (h3o)", ValidWordAbbr(["hello"]).isUnique("hallo"), False)
check("cùng chữ đầu/cuối, khác độ dài", ValidWordAbbr(["dog"]).isUnique("doog"), True)

mẫu: dear                                    PASS (False)
mẫu: cart                                    PASS (True)
mẫu: cane                                    PASS (False)
mẫu: make                                    PASS (True)
mẫu: cake                                    PASS (True)
từ hỏi có trong từ điển nhưng có đồng viết tắt PASS (False)
từ điển trùng lặp                            PASS (True)
trùng lặp + từ khác                          PASS (False)
từ 1 ký tự giống                             PASS (True)
từ 1 ký tự khác                              PASS (True)
từ 2 ký tự                                   PASS (True)
'hello' vs 'hallo' (h3o)                     PASS (False)
cùng chữ đầu/cuối, khác độ dài               PASS (True)


In [17]:
classes = [ValidWordAbbrBrute, ValidWordAbbrSets, ValidWordAbbrOwner, ValidWordAbbrCounter, ValidWordAbbr]
for _ in range(1500):
    d = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 8))]
    objs = [cls(d) for cls in classes]
    for _ in range(5):
        w = rng.choice(d + ["".join(rng.choice("abc") for _ in range(rng.randint(1, 4)))])
        expected = is_unique_by_definition(d, w)
        assert all(o.isUnique(w) == expected for o in objs), (d, w)
print("5 lớp khớp định nghĩa trên 1500 từ điển ngẫu nhiên × 5 truy vấn")

5 lớp khớp định nghĩa trên 1500 từ điển ngẫu nhiên × 5 truy vấn


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "viết tắt có trong từ điển không"

In [18]:
class WrongOnlyAbbr:
    def __init__(self, dictionary):
        self.abbrs = {abbr(w) for w in dictionary}
    def isUnique(self, word):
        return abbr(word) not in self.abbrs

o = WrongOnlyAbbr(["deer", "door", "cake", "card"])
print("cake ->", o.isUnique("cake"), "| đúng = True (chỉ chính nó mang c2e)")

cake -> False | đúng = True (chỉ chính nó mang c2e)


**Sửa:** phải biết **ai** mang viết tắt đó — dùng `owner` hoặc `set`.

### Lỗi 2: Đếm viết tắt trên list có trùng lặp

In [19]:
class WrongCountDup:
    def __init__(self, dictionary):
        self.words = set(dictionary)
        self.count = Counter(abbr(w) for w in dictionary)     # SAI: chưa bỏ trùng
    def isUnique(self, word):
        c = self.count.get(abbr(word), 0)
        return c == 0 or (c == 1 and word in self.words)

print("['cake','cake'], cake ->", WrongCountDup(["cake", "cake"]).isUnique("cake"), "| đúng = True")

['cake','cake'], cake -> False | đúng = True


### Lỗi 3: Sentinel trùng với một từ có thể xuất hiện

Nếu dùng sentinel như `"#"` hay `None` thì vẫn ổn với chữ thường; nhưng dùng một **từ hợp lệ** (ví dụ `"x"`) làm dấu "nhiều" sẽ sai khi truy vấn chính là `"x"`. Chọn sentinel **không thể** là input: `""` (từ có độ dài ≥ 1).

### Lỗi 4: Quên trường hợp `len(word) <= 2`

In [20]:
class WrongShort:
    def __init__(self, dictionary):
        self.owner = {}
        for w in dictionary:
            a = abbr_naive(w)                                  # SAI với từ ngắn
            if a not in self.owner:
                self.owner[a] = w
            elif self.owner[a] != w:
                self.owner[a] = ""
    def isUnique(self, word):
        a = abbr_naive(word)
        return a not in self.owner or self.owner[a] == word

print("abbr_naive('ab') =", abbr_naive("ab"), "| abbr_naive('aab') =", abbr_naive("aab"))
o = WrongShort(["a"])
print("['a'], 'b' -> abbr_naive:", abbr_naive("a"), abbr_naive("b"), "| kết quả:", o.isUnique("b"))

abbr_naive('ab') = a0b | abbr_naive('aab') = a1b
['a'], 'b' -> abbr_naive: a-1a b-1b | kết quả: True


Ở ví dụ này kết quả vẫn đúng, nhưng viết tắt sinh ra (`"a-1a"`, `"a0b"`) **sai định nghĩa**; nếu đề hỏi trả về chính viết tắt hoặc so sánh với chuỗi viết tắt cho sẵn thì lỗi lộ ngay. Luôn cài đúng định nghĩa.

### Lỗi 5: `defaultdict` bị "bẩn" khi truy vấn

`self.groups[abbr(word)]` với `defaultdict(set)` sẽ **thêm** key rỗng mỗi lần hỏi từ lạ — không sai kết quả nhưng tăng bộ nhớ theo số truy vấn. Dùng `.get(...)`.

## 13. Mẹo nhận dạng pattern

- Đề **design** có "khởi tạo một lần, truy vấn nhiều lần" + dữ liệu tĩnh → **tiền xử lý** vào `dict`/`set` trong `__init__`.
- "Nhóm các phần tử theo **khóa chuẩn hóa**" (viết tắt, chuỗi đã sắp xếp, mẫu dịch chuyển…) → `dict: khóa → nhóm`.
- Chỉ cần biết nhóm có 0 / 1 / nhiều phần tử → lưu **đại diện + sentinel** thay vì cả nhóm.
- Từ điển có thể **trùng lặp** → nghĩ ngay tới `set` trước khi đếm.

Bài liên quan: 49 Group Anagrams, 249 Group Shifted Strings, 408 Valid Word Abbreviation, 527 Word Abbreviation, 170 Two Sum III (design).

## 14. Biến thể

### 14.1. Valid Word Abbreviation (408): kiểm tra một viết tắt tổng quát

Viết tắt kiểu `"i12iz4n"` có thể thay **nhiều đoạn** bằng số. Duyệt hai con trỏ, đọc số đầy đủ (không có số 0 đứng đầu).

In [21]:
def valid_word_abbreviation(word: str, ab: str) -> bool:
    i = j = 0
    while i < len(word) and j < len(ab):
        if ab[j].isdigit():
            if ab[j] == "0":
                return False                 # không cho số bắt đầu bằng 0
            num = 0
            while j < len(ab) and ab[j].isdigit():
                num = num * 10 + int(ab[j])
                j += 1
            i += num
        else:
            if word[i] != ab[j]:
                return False
            i += 1
            j += 1
    return i == len(word) and j == len(ab)

print(valid_word_abbreviation("internationalization", "i12iz4n"), valid_word_abbreviation("apple", "a2e"),
      valid_word_abbreviation("internationalization", "i18n"))

True False True


### 14.2. Từ điển **động**: thêm từ sau khi khởi tạo

Với cách `owner`, thêm một từ chỉ là lặp lại logic `__init__` cho từ đó — `O(L)`. (Xóa từ thì cách `owner` không đủ thông tin; cần cách `dict → Counter` các từ.)

In [22]:
class DynamicAbbr(ValidWordAbbrOwner):            # kế thừa lớp đã viết
    def add(self, w: str) -> None:
        a = abbr(w)
        if a not in self.owner:
            self.owner[a] = w
        elif self.owner[a] != w:
            self.owner[a] = ""

o = DynamicAbbr(["cake"])
print(o.isUnique("cane"))
o.add("cane")
print(o.isUnique("cake"), o.isUnique("cane"), o.owner)

False
False False {'c2e': ''}


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Dự đoán:** `dictionary = ["hello", "hallo", "world", "word", "wood"]`. Cho biết `isUnique` của `"hello"`, `"word"`, `"world"`, `"weld"`, `"hi"`.

**Bài 2 — Điền chỗ trống:** hàm `abbr`.

**Bài 3 — Tự viết lại** lớp dùng `dict → set`.

**Bài 4 — Liệt kê nhóm xung đột:** trả về dict chỉ gồm các viết tắt có **≥ 2** từ phân biệt, giá trị là list từ đã sắp xếp.

**Bài 5 — Hỗ trợ xóa từ:** lớp có `remove(w)` (xóa một lần xuất hiện), `isUnique` vẫn đúng. Gợi ý: `dict: abbr → Counter(từ)`.

**Bài 6 — Mini challenge:** viết tắt "**ngắn nhất không mơ hồ**" cho mỗi từ trong một danh sách các từ phân biệt: bắt đầu với `prefix = 1` ký tự (`w[:p] + số + w[-1]`), từ nào còn trùng viết tắt thì tăng `p`. Nếu viết tắt không ngắn hơn từ gốc thì giữ nguyên từ (ý tưởng bài 527).

In [23]:
# Bài 1 — TODO
predict = None   # list bool theo thứ tự ["hello", "word", "world", "weld", "hi"]
# assert predict == [ValidWordAbbr(["hello", "hallo", "world", "word", "wood"]).isUnique(w) for w in ["hello", "word", "world", "weld", "hi"]]

In [24]:
def exercise_abbr(w: str) -> str:
    # TODO: từ ngắn giữ nguyên, còn lại đầu + số + cuối
    pass

# assert [exercise_abbr(w) for w in ["dog", "it", "a", "internationalization"]] == ["d1g", "it", "a", "i18n"]

In [25]:
class ExerciseAbbrSets:
    def __init__(self, dictionary):
        # TODO
        pass
    def isUnique(self, word):
        # TODO
        pass

# o = ExerciseAbbrSets(["deer", "door", "cake", "card"])
# assert [o.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]] == [False, True, False, True, True]

In [26]:
def exercise_conflicts(dictionary: list[str]) -> dict:
    # TODO
    pass

# assert exercise_conflicts(["deer", "door", "cake", "card", "deer"]) == {"d2r": ["deer", "door"]}

In [27]:
class ExerciseRemovable:
    def __init__(self, dictionary):
        # TODO
        pass
    def remove(self, w):
        # TODO
        pass
    def isUnique(self, word):
        # TODO
        pass

# o = ExerciseRemovable(["cake", "cane", "cane"])
# assert o.isUnique("cake") == False; o.remove("cane"); assert o.isUnique("cake") == False; o.remove("cane"); assert o.isUnique("cake") == True

In [28]:
def exercise_shortest_abbr(words: list[str]) -> list[str]:
    # TODO
    pass

# assert exercise_shortest_abbr(["like", "god", "internal", "me", "internet", "interval", "intension", "face", "intrusion"]) == \
#     ["l2e", "god", "internal", "me", "i6t", "interval", "inte4n", "f2e", "intr4n"]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | tính viết tắt từng từ | `h3o`: hello, hallo; `w3d`: chỉ world | `w2d`: word, wood (cả `weld` cũng là `w2d`); `hi`: không ai |
| 2 | `len(w) <= 2` | f-string | `f"{w[0]}{len(w)-2}{w[-1]}"` |
| 3 | xem mục 5 | `setdefault(a, set()).add(w)` | `return S <= {word}` |
| 4 | dùng `dict → set` | lọc `len(S) >= 2` | `sorted(S)` |
| 5 | `dict: abbr → Counter` | xóa: giảm đếm, về 0 thì `del` | unique ⇔ các key của Counter ⊆ `{word}` |
| 6 | nhóm từ theo viết tắt hiện tại | từ nào nhóm có ≥ 2 → tăng prefix | lặp tới khi không còn nhóm trùng |

## 17. Đáp án tham khảo

In [29]:
answer_predict = [False, False, True, False, True]
o = ValidWordAbbr(["hello", "hallo", "world", "word", "wood"])
assert answer_predict == [o.isUnique(w) for w in ["hello", "word", "world", "weld", "hi"]]
print("Bài 1:", answer_predict, "| viết tắt:", {w: abbr(w) for w in ["hello", "word", "world", "weld", "hi"]})

def answer_abbr(w):
    return w if len(w) <= 2 else f"{w[0]}{len(w) - 2}{w[-1]}"

class AnswerAbbrSets:
    def __init__(self, dictionary):
        self.groups = {}
        for w in dictionary:
            self.groups.setdefault(answer_abbr(w), set()).add(w)
    def isUnique(self, word):
        return self.groups.get(answer_abbr(word), set()) <= {word}

def answer_conflicts(dictionary):
    groups = {}
    for w in dictionary:
        groups.setdefault(answer_abbr(w), set()).add(w)
    return {a: sorted(S) for a, S in groups.items() if len(S) >= 2}

class AnswerRemovable:
    def __init__(self, dictionary):
        self.groups = {}
        for w in dictionary:
            self.groups.setdefault(answer_abbr(w), Counter())[w] += 1
    def remove(self, w):
        a = answer_abbr(w)
        c = self.groups.get(a)
        if c and c[w] > 0:
            c[w] -= 1
            if c[w] == 0:
                del c[w]
            if not c:
                del self.groups[a]
    def isUnique(self, word):
        c = self.groups.get(answer_abbr(word))
        return not c or set(c) == {word}

def answer_shortest_abbr(words):
    def make(w, p):
        if len(w) - p <= 2:          # viết tắt không ngắn hơn từ gốc
            return w
        return w[:p] + str(len(w) - p - 1) + w[-1]
    pref = [1] * len(words)
    ans = [make(w, 1) for w in words]
    while True:
        groups = {}
        for i, a in enumerate(ans):
            groups.setdefault(a, []).append(i)
        changed = False
        for idxs in groups.values():
            if len(idxs) > 1:
                for i in idxs:
                    pref[i] += 1
                    ans[i] = make(words[i], pref[i])
                changed = True
        if not changed:
            return ans

assert [answer_abbr(w) for w in ["dog", "it", "a", "internationalization"]] == ["d1g", "it", "a", "i18n"]
o = AnswerAbbrSets(["deer", "door", "cake", "card"])
assert [o.isUnique(w) for w in ["dear", "cart", "cane", "make", "cake"]] == [False, True, False, True, True]
assert answer_conflicts(["deer", "door", "cake", "card", "deer"]) == {"d2r": ["deer", "door"]}
o = AnswerRemovable(["cake", "cane", "cane"])
assert o.isUnique("cake") == False; o.remove("cane"); assert o.isUnique("cake") == False
o.remove("cane"); assert o.isUnique("cake") == True
assert answer_shortest_abbr(["like", "god", "internal", "me", "internet", "interval", "intension", "face", "intrusion"]) == \
    ["l2e", "god", "internal", "me", "i6t", "interval", "inte4n", "f2e", "intr4n"]
print("Tất cả đáp án PASS")

Bài 1: [False, False, True, False, True] | viết tắt: {'hello': 'h3o', 'word': 'w2d', 'world': 'w3d', 'weld': 'w2d', 'hi': 'hi'}
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class ValidWordAbbr:
    def __init__(self, dictionary):
        self.owner = {}
        for w in dictionary:
            a = w if len(w) <= 2 else f"{w[0]}{len(w)-2}{w[-1]}"
            if a not in self.owner:
                self.owner[a] = w
            elif self.owner[a] != w:
                self.owner[a] = ""          # >= 2 từ khác nhau
    def isUnique(self, word):
        a = word if len(word) <= 2 else f"{word[0]}{len(word)-2}{word[-1]}"
        return a not in self.owner or self.owner[a] == word
```

**Câu thần chú:** *"Unique ⇔ tập từ cùng viết tắt ⊆ {word}."*

**Checklist:**
- [ ] Viết tắt từ dài ≤ 2 là chính nó.
- [ ] Từ điển có thể trùng lặp — bỏ trùng trước khi đếm.
- [ ] Từ hỏi nằm sẵn trong từ điển vẫn có thể `True`.
- [ ] Sentinel không trùng với từ hợp lệ (`""`).
- [ ] Tiền xử lý trong `__init__`, truy vấn `O(L)`.
- [ ] Tên lớp `ValidWordAbbr`, method `isUnique`.